# Reality Monitoring Task — Parsers across Memory & Feedback configurations

Walks through how the same Reality-Monitoring (RM) word-pair paradigm is run under four different psychscanner configurations:

| # | Variant | `chain_type` | `memory` | `feedback` | Parser |
|---|---------|--------------|----------|-----------|--------|
| 1 | Single-turn               | `item`  | `SingleTurn` | off | `ResponseRmStEI`   |
| 2 | Trial chain               | `trial` | `Convo`      | off | `AllResponseRMEI`  |
| 3 | Episodic conversation, no feedback | `task`  | `Convo`      | off | `TwoResponses`     |
| 4 | Episodic conversation + feedback   | `task`  | `Convo`      | on  | `TwoResponses`     |

All four task JSONs were trimmed from the rmllm dissertation repo's `data/raw/run_tasks/` and copied into `examples/tasks/`. The originals (`rm_2op.json`, `rm_2op_tc.json`, `rm_2op_convo.json`, `rm_2op_convo_feedback_allparts.json`) have 40–144 trial groups each; the demos here keep only 2–4 to make the notebook runnable in a couple of minutes.

## Why these four?

Each variant exercises a different intersection of memory, parser shape, and feedback wiring:

- **Single-turn** is the baseline: every word pair is judged independently.
- **Trial chain** breaks one judgment into four sub-steps (Word_2 → relatedness → source judgment → confidence) routed through a Pydantic `Union`.
- **Episodic conversation** lets the model see prior turns within the task and uses a per-trial `system_message`.
- **Feedback** adds a `feedback_fn` that injects ground-truth source information between trials so the model can update its model of what's going on.

## 1. Setup

Requires Ollama running locally with a small instruct model. Defaults match the other tutorials in this folder.

In [1]:
import ast
import json
import shutil
from pathlib import Path

from langchain_core.messages import HumanMessage
from psychscanner import ExpCard, ExpCardInit, ScannerModel
from psychscanner.parsers import (
    list_parsers,
    get_parser,
    ResponseRmStEI,
    AllResponseRMEI,
    TwoResponses,
)

TASKS_DIR = Path.cwd() / 'tasks'
RUN_DIR   = Path.cwd() / '_rm_tutorial_runs'
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)

MODEL_NAME   = 'smollm2:360m-instruct-fp16'
MODEL_FAMILY = 'ollama'

task_files = {
    'singleturn'  : TASKS_DIR / 'rm_singleturn_demo.json',
    'trialchain'  : TASKS_DIR / 'rm_trialchain_demo.json',
    'episodic'    : TASKS_DIR / 'rm_episodic_demo.json',
    'episodic_fb' : TASKS_DIR / 'rm_episodic_fb_demo.json',
}
for name, p in task_files.items():
    print(f'  {name:13} -> {p.name}  (exists={p.exists()})')

  singleturn    -> rm_singleturn_demo.json  (exists=True)
  trialchain    -> rm_trialchain_demo.json  (exists=True)
  episodic      -> rm_episodic_demo.json  (exists=True)
  episodic_fb   -> rm_episodic_fb_demo.json  (exists=True)


## 2. Inspect the four task JSONs

Each task declares its own parser name (resolved through the `psychscanner.parsers` registry when `parser='1'` is set on the card).

In [2]:
print(f'{"variant":13} {"taskname":24} {"tasktype":18} {"chain_type":11} {"parser":18} items')
print('-' * 100)
for name, p in task_files.items():
    d = json.loads(p.read_text())
    print(f'{name:13} {d["taskname"]:24} {d["tasktype"]:18} {d["chain_type"]:11} {d["parser"]:18} {len(d["items"]):>5}')

variant       taskname                 tasktype           chain_type  parser             items
----------------------------------------------------------------------------------------------------
singleturn    zs_rm_2op                sc                 item        ResponseRmStEI         4
trialchain    zs_rm_2op_tc             sc                 trial       AllResponseRMEI        2
episodic      rm_2op_convo_nofb        episodic_system    task        TwoResponses           4
episodic_fb   rm_2op_convo_fb          episodic_system    task        TwoResponses           4


## 3. The RM-relevant parsers

All three parsers used by these variants live in `parser_tasks.py` (paradigm-specific) — except `TwoResponses`, which is in `parser_general.py` because it's a generic two-string container that the episodic task happens to reuse.

In [3]:
for cls in (ResponseRmStEI, AllResponseRMEI, TwoResponses):
    print(f'{cls.__name__}  (defined in {cls.__module__.split(".")[-1]})')
    for fname, fld in cls.model_fields.items():
        ann = str(fld.annotation).replace('typing.', '')
        print(f'  - {fname}: {ann}')
    print()

ResponseRmStEI  (defined in parser_tasks)
  - Word_2: <class 'str'>
  - Rating: <class 'float'>
  - Judgment: Literal['external', 'internal']
  - Confidence: Literal[1, 2, 3, 4, 5, 6]

AllResponseRMEI  (defined in parser_tasks)
  - response: Union[psychscanner.datasets.prompts.parser_tasks.Word2, psychscanner.datasets.prompts.parser_tasks.RelatednessRating, psychscanner.datasets.prompts.parser_tasks.JudgmentEI, psychscanner.datasets.prompts.parser_tasks.Confidence16]

TwoResponses  (defined in parser_general)
  - Response_1: <class 'str'>
  - Response_2: <class 'str'>



## 4. A small helper to run a card and collect parsed responses

Pulls the AIMessage `pred_resp.content` (a stringified dict, current behavior) and parses it back with `ast.literal_eval`.

In [4]:
def make_card(variant, *, memory, chain_type, feedback='0', feedback_fn=None):
    card = ExpCardInit()
    card.proj_dir       = RUN_DIR / variant
    card.projectname    = variant
    card.model          = MODEL_NAME
    card.family         = MODEL_FAMILY
    card.parameters     = {'temperature': 0}
    card.task_file      = task_files[variant]
    card.parser         = '1'              # resolved from the task JSON via the registry
    card.cogtype        = 'no'
    card.nsim           = 1
    card.tunnel_status  = '0'
    card.memory         = memory
    card.chain_type     = chain_type
    card.feedback       = feedback
    card.feedback_fn    = feedback_fn
    return card

def parse_pred_resp(pred_resp):
    """Decode the stringified-dict AIMessage content back to a Python object."""
    content = pred_resp.content if hasattr(pred_resp, 'content') else str(pred_resp)
    try:
        return ast.literal_eval(content)
    except Exception:
        return {'_raw': content}

def show_trials(trials, max_stim_chars=40):
    print(f'  {"trcode":15}  parsed response')
    print(f'  {"-"*15}  {"-"*60}')
    for t in trials:
        parsed = parse_pred_resp(t['pred_resp'])
        print(f'  {t["trcode"]:15}  {parsed}')

## 5. Variant 1 — Single-turn (`chain_type=item`, no memory)

Each word pair is parsed by `ResponseRmStEI` into a structured dict with `Word_2`, `Rating` (0–100), `Judgment` (`external`/`internal`), and `Confidence` (1–6). Trials are independent — the model sees only the current pair.

In [5]:
card_v1 = make_card('singleturn', memory='SingleTurn', chain_type='item')
exp_v1  = ExpCard(card_v1)
print(f'Resolved parser: {exp_v1.parser.__name__}  (in {exp_v1.parser.__module__.split(".")[-1]})')
scanner_v1 = ScannerModel(expcard=exp_v1)
trials_v1 = scanner_v1.run()[0]
print()
show_trials(trials_v1)

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_rm_tutorial_runs/singleturn


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_rm_tutorial_runs/singleturn/singleturn/zs_rm_2op/ollama_smollm2:360m-instruct-fp16_SingleTurn


----<>----


Resolved parser: ResponseRmStEI  (in parser_tasks)
{'temperature': 0}
--<chat model>-- model='smollm2:360m-instruct-fp16' temperature=0.0


...

4it [00:51, 12.86s/it]


2026-05-03 19:20:52.428 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-05-03 19:20:52.433 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END



  trcode           parsed response
  ---------------  ------------------------------------------------------------
  imagined_1       {'Word_2': '<your word based on the rules for word-pair.>', 'Rating': 3.0, 'Judgment': 'internal', 'Confidence': 1}
  perceived_2      {'Word_2': '<your word based on the rules for word-pair.>', 'Rating': 3.0, 'Judgment': 'internal', 'Confidence': 1}
  imagined_3       {'Word_2': '<your word based on the rules for word-pair.>', 'Rating': 3.0, 'Judgment': 'internal', 'Confidence': 1}
  perceived_4      {'Word_2': '<your word based on the rules for word-pair.>', 'Rating': 3.0, 'Judgment': 'internal', 'Confidence': 4}


## 6. Variant 2 — Trial chain (`chain_type=trial`, multi-stimulus per trial)

Each item key (e.g. `imagined_1`) holds **4 sequential stimuli** that walk the model through (1) identifying word_2, (2) rating relatedness, (3) judging source, (4) reporting confidence. The parser is `AllResponseRMEI = Union[Word2, RelatednessRating, JudgmentEI, Confidence16]` — the model picks the right component shape for each step.

With `chain_type='trial'` and `memory='Convo'`, prior steps within the *same* trial are visible to the model; trials don't see each other.

In [6]:
card_v2 = make_card('trialchain', memory='Convo', chain_type='trial')
exp_v2  = ExpCard(card_v2)
print(f'Resolved parser: {exp_v2.parser.__name__}  (in {exp_v2.parser.__module__.split(".")[-1]})')
scanner_v2 = ScannerModel(expcard=exp_v2)
trials_v2 = scanner_v2.run()[0]
print()
show_trials(trials_v2)

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_rm_tutorial_runs/trialchain


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_rm_tutorial_runs/trialchain/trialchain/zs_rm_2op_tc/ollama_smollm2:360m-instruct-fp16_Convo


----<>----


Resolved parser: AllResponseRMEI  (in parser_tasks)
{'temperature': 0}
--<chat model>-- model='smollm2:360m-instruct-fp16' temperature=0.0


...

8it [00:44,  5.59s/it]


2026-05-03 19:21:37.280 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-05-03 19:21:37.285 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END



  trcode           parsed response
  ---------------  ------------------------------------------------------------
  imagined_1       {'response': {'Word_2': 'sandpaper'}}
  imagined_1       {'response': {'Relatedness_Rating': 100.0}}
  imagined_1       {'response': {'Judgment': 'external'}}
  imagined_1       {'response': {'Confidence': 5}}
  perceived_2      {'response': {'Word_2': 'sweet'}}
  perceived_2      {'response': {'Relatedness_Rating': 100.0}}
  perceived_2      {'response': {'Judgment': 'external'}}
  perceived_2      {'response': {'Confidence': 5}}


## 7. Variant 3 — Episodic conversation, no feedback (`chain_type=task`)

Now we use `tasktype='episodic_system'` so the per-trial `system_message` overrides the task-level one. Memory is `Convo` and chain is `task`, so all trials share a single LangGraph thread — the model sees its own previous responses as conversation history. No feedback is injected, so the model cannot tell whether its prior judgments were right.

In [7]:
card_v3 = make_card('episodic', memory='Convo', chain_type='task')
exp_v3  = ExpCard(card_v3)
print(f'Resolved parser: {exp_v3.parser.__name__}  (in {exp_v3.parser.__module__.split(".")[-1]})')
scanner_v3 = ScannerModel(expcard=exp_v3)
trials_v3 = scanner_v3.run()[0]
print()
show_trials(trials_v3)

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_rm_tutorial_runs/episodic


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_rm_tutorial_runs/episodic/episodic/rm_2op_convo_nofb/ollama_smollm2:360m-instruct-fp16_Convo


----<>----


Resolved parser: TwoResponses  (in parser_general)
{'temperature': 0}
--<chat model>-- model='smollm2:360m-instruct-fp16' temperature=0.0


...

4it [00:36,  9.10s/it]


2026-05-03 19:22:13.789 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-05-03 19:22:13.790 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END



  trcode           parsed response
  ---------------  ------------------------------------------------------------
  imagined_1       {'Response_1': "You are a helpful participant performing a task with two different components for successful response. Each component refers to a unique problem about the task as described in the instructions below. Two components of the task are: ['WORD_PAIR_TASK', 'RELATEDNESS_RATING_BETWEEN_WORD_1_AND_WORD_2'].", 'Response_2': 'You are **prohibted** to imagine:**'}
  perceived_2      {'Response_1': "You are a helpful participant performing a task with two different components for successful response. Each component refers to a unique problem about the task as described in the instructions below. Two components of the task are: ['WORD_PAIR_TASK', 'RELATEDNESS_RATING_BETWEEN_WORD_1_AND_WORD_2'].", 'Response_2': 'You are **prohibted** to imagine:**'}
  imagined_3       {'Response_1': "You are a helpful participant performing a task with two different co

## 8. Variant 4 — Episodic conversation **with feedback**

Same task structure as variant 3, but now `feedback='1'` and we provide a `feedback_fn`. Between each trial, our class injects a corrective signal: "the previous word pair was actually internally generated" (or externally perceived). The model carries that into the next trial via `Convo` memory.

### The 3-method `feedback_fn` contract (current package design)

psychscanner instantiates a fresh feedback object **per trial** and calls two methods on it:

1. `update_trial_stim(finput, fb_response)` — runs *before* the LLM call. Lets the user mutate the next trial's input dict using the previous trial's feedback.
2. `generate_feedback(trdata, pred_dict, input_dict, parser_status, trial_item_collector)` — runs *after* the LLM call. Returns a feedback string that will be passed to the *next* trial's `update_trial_stim`.

(This contract is on the planning list to be replaced with simpler `pre_trial`/`post_trial` callbacks; for now we use the existing shape.)

In [8]:
class SimpleRMFeedback:
    """Per-trial feedback injector for RM word-pair tasks.

    Strategy: after each trial, look at the trcode prefix to determine the
    ground-truth source of the second word (``imagined_*`` -> internally
    generated; ``perceived_*`` -> externally perceived) and emit a sentence
    telling the model what the truth was. The next trial's input gets that
    sentence appended as a HumanMessage before the model is invoked.
    """

    def __init__(self, trial_data):
        self.trial_data = trial_data

    def update_trial_stim(self, finput, fb_response):
        """Inject prior feedback into the upcoming trial's input."""
        if not fb_response:
            return finput
        new_inputs = list(finput['inputs']) + [HumanMessage(f'FEEDBACK: {fb_response}')]
        return {**finput, 'inputs': new_inputs}

    def generate_feedback(self, trdata, pred_dict, input_dict, parser_status, trial_item_collector):
        trcode = trdata.get('trcode', '')
        if 'imagined' in trcode:
            truth = 'internally generated (imagined)'
        elif 'perceived' in trcode:
            truth = 'externally perceived'
        else:
            return None
        return f'For the word pair you just responded to ({trcode}), the second word was {truth}.'


card_v4 = make_card(
    'episodic_fb',
    memory='Convo', chain_type='task',
    feedback='1', feedback_fn=SimpleRMFeedback,
)
exp_v4  = ExpCard(card_v4)
print(f'Resolved parser: {exp_v4.parser.__name__}  (in {exp_v4.parser.__module__.split(".")[-1]})')
scanner_v4 = ScannerModel(expcard=exp_v4)
trials_v4 = scanner_v4.run()[0]
print()
show_trials(trials_v4)

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_rm_tutorial_runs/episodic_fb


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_rm_tutorial_runs/episodic_fb/episodic_fb/rm_2op_convo_fb/ollama_smollm2:360m-instruct-fp16_Convo


----<>----


Resolved parser: TwoResponses  (in parser_general)
{'temperature': 0}
--<chat model>-- model='smollm2:360m-instruct-fp16' temperature=0.0


...

4it [00:59, 14.92s/it]


2026-05-03 19:23:13.529 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-05-03 19:23:13.532 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END



  trcode           parsed response
  ---------------  ------------------------------------------------------------
  imagined_1       {'Response_1': 'You are a helpful participant performing a task with two different components for successful response. You have made the commitment to make sure to follow all the the instructions for each of the components and formatting your task response. You will also be provided with feedback on previous trials in the word-pair task. Try to use the feedback to improve your performance on the task.', 'Response_2': "You are a helpful participant performing a task with two different components for successful response. Each component refers to a unique problem about the task as described in the instructions below. Two components of the task are: ['WORD_PAIR_TASK', 'RELATEDNESS_RATING_BETWEEN_WORD_1_AND_WORD_2'] You follow all the instructions related to different components of the task to give accurate response. Try to be as accurate as possible."}
  pe

## 9. Side-by-side summary

Same word pairs, four configurations. Notice how the parsed shape changes per variant (because each task uses a different parser), and how Convo + feedback can shift the model's later responses relative to no-feedback.

In [9]:
summary = {
    'V1 singleturn / item / no fb'    : trials_v1,
    'V2 trialchain / trial / no fb'   : trials_v2,
    'V3 episodic / task / no fb'      : trials_v3,
    'V4 episodic / task / +feedback'  : trials_v4,
}
for label, trials in summary.items():
    print(f'=== {label} ({len(trials)} trials) ===')
    for t in trials:
        parsed = parse_pred_resp(t['pred_resp'])
        print(f'  {t["trcode"]:15} -> {parsed}')
    print()

=== V1 singleturn / item / no fb (4 trials) ===
  imagined_1      -> {'Word_2': '<your word based on the rules for word-pair.>', 'Rating': 3.0, 'Judgment': 'internal', 'Confidence': 1}
  perceived_2     -> {'Word_2': '<your word based on the rules for word-pair.>', 'Rating': 3.0, 'Judgment': 'internal', 'Confidence': 1}
  imagined_3      -> {'Word_2': '<your word based on the rules for word-pair.>', 'Rating': 3.0, 'Judgment': 'internal', 'Confidence': 1}
  perceived_4     -> {'Word_2': '<your word based on the rules for word-pair.>', 'Rating': 3.0, 'Judgment': 'internal', 'Confidence': 4}

=== V2 trialchain / trial / no fb (8 trials) ===
  imagined_1      -> {'response': {'Word_2': 'sandpaper'}}
  imagined_1      -> {'response': {'Relatedness_Rating': 100.0}}
  imagined_1      -> {'response': {'Judgment': 'external'}}
  imagined_1      -> {'response': {'Confidence': 5}}
  perceived_2     -> {'response': {'Word_2': 'sweet'}}
  perceived_2     -> {'response': {'Relatedness_Rating': 100.0

## What this tutorial demonstrates

1. **Same paradigm, four wirings.** All four variants are RM word-pair tasks; differences come from `chain_type`, `memory`, `feedback`, and the parser declared in the task JSON.
2. **`parser='1'` resolution** through the registry works for every parser used here (`ResponseRmStEI` and `AllResponseRMEI` from `parser_tasks`, `TwoResponses` from `parser_general`).
3. **`chain_type` controls thread persistence.** `item` is stateless. `trial` shares a thread across the multi-step stimuli of *one* trial. `task` shares a single thread across *all* trials in the task — that's what makes episodic conversation possible.
4. **Feedback wiring is explicit.** `feedback='1'` plus a `feedback_fn` class instance is enough to splice ground-truth signal between trials. The class only needs `__init__`, `update_trial_stim`, and `generate_feedback` — no inheritance.

## Caveats

- The feedback contract reinstantiates the class **every trial**. Cross-trial state has to live in the closure / module scope, not on `self`. This is on the planning list to fix.
- Parsed responses are stringified before being put into `Convo` history. The model in `Convo` mode therefore sees its own dict-shaped prior responses as text, not natural language. Also on the planning list (item P5).